# C3 · The optimism of best_score_: measuring it with a simulation

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Measure, on 16 simulated datasets with a weak signal, by how much the `best_score_` of a `GridSearchCV` exceeds the true performance of the
chosen model, and verify that a nested cross-validation estimates it without bias.

## Dataset

`generate(seed)` produces, with `make_classification` (2 informative features out of 15, noise in the labels), a **development** dataset of 120
rows and a **truth** one of 2000 rows never used to choose anything. Grid of 24 combinations for a KNN (number of neighbors, weights, metric).

## Educational purpose

Turn "best_score_ is optimistic" from a sentence into a number. You will see that the estimate grows with the number of combinations tried and that
nested cross-validation corrects it, at the cost of many trainings.

## Task

1. Write `experiment(seed)`: generate the data, run `GridSearchCV(estimator, grid, cv=StratifiedKFold(5, shuffle=True, random_state=seed), scoring="roc_auc")` on the development set and return the triple `(best_score, true_auc, nested_estimate)`. `true_auc` is the AUC of the `best_estimator_` on the truth dataset; `nested_estimate` is `cross_val_score(GridSearchCV(..., cv=3 folds), X_dev, y_dev, cv=5 outer folds, scoring="roc_auc").mean()`.
2. Run `experiment` for `seed` from 0 to 15 and build `results` (DataFrame, columns `best_score`, `true_auc`, `nested`).
3. Compute `best_optimism = (best_score − true_auc).mean()` and `nested_error = (nested − true_auc).mean()`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

estimator = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid = {"kneighborsclassifier__n_neighbors": [1, 5, 11, 17, 25, 35],
        "kneighborsclassifier__weights": ["uniform", "distance"],
        "kneighborsclassifier__p": [1, 2]}                       # 6 x 2 x 2 = 24 combinations


def generate(seed):
    X, y = make_classification(n_samples=2120, n_features=15, n_informative=2, n_redundant=0, n_clusters_per_class=1,
                               class_sep=0.35, flip_y=0.2, random_state=seed)
    return X[:120], y[:120], X[120:], y[120:]                       # development (120) and truth (2000)

## Your work

In [ ]:
def experiment(seed):
    X_dev, y_dev, X_true, y_true = generate(seed)
    search = ...
    best_score = ...
    true_auc = ...
    nested_estimate = ...
    return best_score, true_auc, nested_estimate

In [ ]:
results = pd.DataFrame([experiment(s) for s in range(16)], columns=["best_score", "true_auc", "nested"])
best_optimism = ...
nested_error = ...
print(results.mean().round(3).to_dict())

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`search = GridSearchCV(estimator, grid, cv=StratifiedKFold(5, shuffle=True, random_state=seed), scoring="roc_auc").fit(X_dev, y_dev)`; `roc_auc_score(y_true, search.predict_proba(X_true)[:, 1])`. Nested: `cross_val_score(GridSearchCV(estimator, grid, cv=3, scoring="roc_auc"), X_dev, y_dev, cv=StratifiedKFold(5, shuffle=True, random_state=seed), scoring="roc_auc").mean()`.'):
    check(results.shape == (16, 3), "16 experiments", "one row per seed")
    check((results["best_score"] > results["true_auc"]).mean() > 0.8, "best_score exceeds the true performance in the great majority of the datasets",
          "in over 80% of the datasets best_score_ should exceed the AUC on the truth dataset")
    check(close(best_optimism, (results.best_score - results.true_auc).mean()) and best_optimism > 0.04,
          "the mean optimism of best_score_ exceeds 0.04 of AUC", "optimism = mean(best_score - true_auc)")
    check(close(nested_error, (results.nested - results.true_auc).mean()) and abs(nested_error) < best_optimism / 2,
          "the nested estimate is much less biased than best_score_ (less than half of the optimism)", "error = mean(nested - true_auc)")
done()

---
The commented solution is in `solutions/C_break_the_model/C3_best_score_optimism.ipynb`. Open it after you pass the check, or after you have really tried.